# Remaining-seed SIA/PIC experiment

Kaggle execution notebook for seeds 3 and 5. It trains and probes all four arms with attention-KL off, then trains and probes the SIA-containing arms with attention-KL on.


## 1. Repository bootstrap


In [1]:
from pathlib import Path
import os, sys, shutil, zipfile

work_root = Path("/kaggle/working")
current = Path.cwd()
if (current / "run_experiments.py").exists():
    repo_root = current
else:
    repo_root = work_root / "mipic"
    candidates = list(Path("/kaggle/input").rglob("run_experiments.py")) if Path("/kaggle/input").exists() else []
    if candidates:
        source = candidates[0].parent
        if repo_root.exists():
            shutil.rmtree(repo_root)
        shutil.copytree(source, repo_root, ignore=shutil.ignore_patterns("results", "__pycache__", ".pytest_cache", "*.pyc"))
    else:
        raise FileNotFoundError("MIPIC repository not found under /kaggle/input")
os.chdir(repo_root)
sys.path.insert(0, str(repo_root))
os.environ["PYTHONPATH"] = str(repo_root)
print("Repository:", repo_root)


Repository: /kaggle/working/mipic


## 2. Install dependencies


In [2]:
!pip install -q -r requirements.txt

## 3. Hardware


In [3]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(" ", index, torch.cuda.get_device_name(index))


PyTorch: 2.11.0+cu130 | CUDA: True | GPUs: 2
  0 Tesla T4
  1 Tesla T4


## 4. Locate datasets


In [4]:
import csv
import io
import pandas as pd
import urllib.request

def first_existing(candidates, local_candidates=()):
    for candidate in local_candidates:
        path = repo_root / candidate
        if path.exists():
            return path
    for candidate in candidates:
        hits = sorted(Path("/kaggle/input").rglob(candidate))
        if hits:
            return hits[0]
    return None

corpus = first_existing(["final_data.csv"], ["data/final_data.csv"])
tweet_train = first_existing(["tweet_train.csv"], ["data/tweet_train.csv"])
tweet_test = first_existing(["tweet_test.csv"], ["data/tweet_test.csv"])
banking_train = first_existing(["banking77_train.csv", "banking_train.csv"], ["data/banking/banking_train.csv"])
banking_test = first_existing(["banking77_test.csv", "banking_test.csv"], ["data/banking/banking77_test.csv"])
stsb_test = first_existing(["stsb_test.csv", "sts_test.csv"], ["data/stsb_test.csv"])
if stsb_test is None:
    with urllib.request.urlopen("https://dl.fbaipublicfiles.com/glue/data/STS-B.zip", timeout=60) as response:
        archive = zipfile.ZipFile(io.BytesIO(response.read()))
    with archive.open("STS-B/dev.tsv") as handle:
        parsed = list(csv.reader(io.StringIO(handle.read().decode("utf-8")), delimiter="\t"))
    rows = [{"sentence1": fields[7], "sentence2": "\t".join(fields[8:-1]), "score": fields[-1]}
            for fields in parsed[1:] if len(fields) >= 10]
    stsb_test = work_root / "stsb_validation_remaining.csv"
    pd.DataFrame(rows)[["sentence1", "sentence2", "score"]].to_csv(stsb_test, index=False)
required = {"training corpus": corpus, "tweet train": tweet_train, "tweet test": tweet_test,
            "Banking77 train": banking_train, "Banking77 test": banking_test, "STS-B": stsb_test}
missing = [name for name, path in required.items() if path is None]
if missing:
    raise FileNotFoundError("Missing: " + ", ".join(missing))
for name, path in required.items():
    print(f"{name}: {path} ({len(pd.read_csv(path)):,} rows)")
for name, path in [("tweet train", tweet_train), ("tweet test", tweet_test),
                   ("Banking77 train", banking_train), ("Banking77 test", banking_test)]:
    columns = set(pd.read_csv(path, nrows=0).columns)
    if not {"text", "label"}.issubset(columns):
        raise ValueError(f"{name} must contain text and label columns; got {sorted(columns)}")
if not {"sentence1", "sentence2", "score"}.issubset(set(pd.read_csv(stsb_test, nrows=0).columns)):
    raise ValueError("STS-B must contain sentence1, sentence2, score columns")


training corpus: /kaggle/working/mipic/data/final_data.csv (20,240 rows)
tweet train: /kaggle/working/mipic/data/tweet_train.csv (26,732 rows)
tweet test: /kaggle/working/mipic/data/tweet_test.csv (3,432 rows)
Banking77 train: /kaggle/working/mipic/data/banking/banking_train.csv (10,003 rows)
Banking77 test: /kaggle/working/mipic/data/banking/banking77_test.csv (3,076 rows)
STS-B: /kaggle/working/stsb_validation_remaining.csv (1,468 rows)


## 5. Test gate


In [5]:
!python -m compileall -q . && echo "compileall OK"
!python -m pytest -q tests/test_run_experiments.py tests/test_checkpoint_safety.py


compileall OK
.......                                                                  [100%]
7 passed in 20.46s


In [6]:
print("Implementation tests passed; the experiment launch is in the next section.")

Implementation tests passed; the experiment launch is in the next section.


## 6. Execution configuration


In [7]:
SEEDS = [3, 5]
OUTPUT_DIR = "/kaggle/working/sia_pic_remaining_seeds"
FRESH = False
os.environ["MIPIC_EPOCHS"] = "5"
os.environ["MIPIC_RUN_MODE"] = "ddp"
os.environ["MIPIC_LEARNING_RATE"] = "2e-5"
os.environ["MIPIC_BATCH_SIZE"] = "8"
os.environ["MIPIC_MAX_LENGTH"] = "256"
os.environ["MIPIC_NUM_WORKERS"] = "0"
os.environ["MIPIC_LOG_EVERY"] = "50"
os.environ["MIPIC_STACK_DUMP_SECONDS"] = "900"
os.environ["PYTHONUTF8"] = "1"
import shutil
if not torch.cuda.is_available() or torch.cuda.device_count() < 2:
    raise RuntimeError(f"This run requires 2 CUDA GPUs for DDP; found {torch.cuda.device_count()}")
if shutil.which("torchrun") is None:
    raise RuntimeError("torchrun is not available in the Kaggle environment")
print("seeds:", SEEDS)
print("output:", OUTPUT_DIR)
print("settings: epochs=5, lr=2e-5, batch=16 global, alpha=0.4")
print("GPUs:", torch.cuda.device_count(), "torchrun:", shutil.which("torchrun"))
print("matrix: 2 seeds x (4 KL-off arms + 2 KL-on arms) = 12 training runs")


seeds: [3, 5]
output: /kaggle/working/sia_pic_remaining_seeds
settings: epochs=5, lr=2e-5, batch=16 global, alpha=0.4
GPUs: 2 torchrun: /usr/local/bin/torchrun
matrix: 2 seeds x (4 KL-off arms + 2 KL-on arms) = 12 training runs


## 7. Launch status


In [8]:
print("Output directory:", OUTPUT_DIR)

Output directory: /kaggle/working/sia_pic_remaining_seeds


## 8. Full experiment


In [9]:
import subprocess
command = [
    sys.executable, str(repo_root / "experiments" / "sia_pic" / "experiment.py"),
    "--corpus", str(corpus), "--tweet-train", str(tweet_train), "--tweet-test", str(tweet_test),
    "--banking-train", str(banking_train), "--banking-test", str(banking_test),
    "--stsb-test", str(stsb_test), "--output-dir", OUTPUT_DIR,
    "--seeds", *[str(seed) for seed in SEEDS],
]
if FRESH:
    command.append("--fresh")
env = os.environ.copy()
env["MIPIC_FRESH"] = "1" if FRESH else "0"
print("Launching:", " ".join(command))
subprocess.run(command, check=True, env=env)


Launching: /usr/bin/python3 /kaggle/working/mipic/experiments/sia_pic/experiment.py --corpus /kaggle/working/mipic/data/final_data.csv --tweet-train /kaggle/working/mipic/data/tweet_train.csv --tweet-test /kaggle/working/mipic/data/tweet_test.csv --banking-train /kaggle/working/mipic/data/banking/banking_train.csv --banking-test /kaggle/working/mipic/data/banking/banking77_test.csv --stsb-test /kaggle/working/stsb_validation_remaining.csv --output-dir /kaggle/working/sia_pic_remaining_seeds --seeds 3 5


W1109 19:50:50.107000 98 torch/distributed/run.py:851] 
W1109 19:50:50.107000 98 torch/distributed/run.py:851] *****************************************
W1109 19:50:50.107000 98 torch/distributed/run.py:851] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W1109 19:50:50.107000 98 torch/distributed/run.py:851] *****************************************


DDP: nccl, world_size=2, collective timeout=20 min, NCCL_P2P_DISABLE=1, NCCL_SHM_DISABLE=<unset>
Seed: 3 (rank-adjusted per process; this rank uses 3)
Device: cuda:0 | distributed_world_size=2 | GPU: Tesla T4

STARTING EXPERIMENT: MRL (TRAINING & EVALUATION)

STARTING EXPERIMENT: MRL (TRAINING & EVALUATION)
Training corpus: 20,240 rows; sources={'BANKING77': 3000, 'TWEET_EVAL_SENT': 3000, 'DEEHAN1866/WIC': 3000, 'MRPC': 3000, 'DGEM_FORMAT': 3000, 'STSB': 2893, 'SICK': 2347}
DataLoader: rows=20,240, per_rank_batch=8, world_size=2, effective_global_batch=16, sampler=DistributedSampler


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14558.65it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect i

DDP: static_graph=True (deterministic reducer ordering)
[rank 0] 19:51:07 epoch 1/5: batch 1/1265[rank 1] 19:51:07 epoch 1/5: batch 1/1265

[rank 1] 19:51:16 epoch 1/5: batch 50/1265  |g|=159.100
[rank 0] 19:51:16 epoch 1/5: batch 50/1265  |g|=159.100
[rank 0] 19:51:22 epoch 1/5: batch 100/1265  |g|=25.592
[rank 1] 19:51:22 epoch 1/5: batch 100/1265  |g|=25.592
[rank 1] 19:51:29 epoch 1/5: batch 150/1265  |g|=15.767
[rank 0] 19:51:29 epoch 1/5: batch 150/1265  |g|=15.767
[rank 1] 19:51:36 epoch 1/5: batch 200/1265  |g|=26.499
[rank 0] 19:51:36 epoch 1/5: batch 200/1265  |g|=26.499
[rank 0] 19:51:43 epoch 1/5: batch 250/1265  |g|=18.594
[rank 1] 19:51:43 epoch 1/5: batch 250/1265  |g|=18.594
[rank 1] 19:51:50 epoch 1/5: batch 300/1265  |g|=2.536
[rank 0] 19:51:50 epoch 1/5: batch 300/1265  |g|=2.536
[rank 0] 19:51:57 epoch 1/5: batch 350/1265  |g|=5.377
[rank 1] 19:51:57 epoch 1/5: batch 350/1265  |g|=5.377
[rank 0] 19:52:03 epoch 1/5: batch 400/1265  |g|=7.214
[rank 1] 19:52:03 epoch 1

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14720.87it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect i

... (49 outputs left out)

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)
 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)
Training corpus: 20,240 rows; sources={'BANKING77': 3000, 'TWEET_EVAL_SENT': 3000, 'DEEHAN1866/WIC': 3000, 'MRPC': 3000, 'DGEM_FORMAT': 3000, 'STSB': 2893, 'SICK': 2347}
DataLoader: rows=20,240, per_rank_batch=8, world_size=2, effective_global_batch=16, sampler=DistributedSampler
DDP: static_graph=True (deterministic reducer ordering)
[rank 0] 01:34:45 epoch 1/5: batch 1/1265
[rank 1] 01:34:45 epoch 1/5: batch 1/1265
[rank 1] 01:35:05 epoch 1/5: batch 50/1265  |g|=74.471
[rank 0] 01:

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6933.96it/s]
[transformers] BertModel LOAD REPORT from: huawei-noah/TinyBERT_General_6L_768D
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
fit_denses.{0, 1, 2, 3, 4, 5, 6}.bias      | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
fit_denses.{0, 1, 2, 3, 4, 5, 6}.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect id

 PICModule initialized with 5 pipeline pairs:
   (layer_1, dim_16) ? (layer_2, dim_32)
   (layer_2, dim_32) ? (layer_3, dim_64)
   (layer_3, dim_64) ? (layer_4, dim_256)
   (layer_4, dim_256) ? (layer_5, dim_512)
   (layer_5, dim_512) ? (layer_6, dim_768)
Saved scores and contrasts under /kaggle/working/sia_pic_remaining_seeds


CompletedProcess(args=['/usr/bin/python3', '/kaggle/working/mipic/experiments/sia_pic/experiment.py', '--corpus', '/kaggle/working/mipic/data/final_data.csv', '--tweet-train', '/kaggle/working/mipic/data/tweet_train.csv', '--tweet-test', '/kaggle/working/mipic/data/tweet_test.csv', '--banking-train', '/kaggle/working/mipic/data/banking/banking_train.csv', '--banking-test', '/kaggle/working/mipic/data/banking/banking77_test.csv', '--stsb-test', '/kaggle/working/stsb_validation_remaining.csv', '--output-dir', '/kaggle/working/sia_pic_remaining_seeds', '--seeds', '3', '5'], returncode=0)

## 9. Results


In [10]:
import json
output = Path(OUTPUT_DIR)
scores = output / "scores.json"
contrasts = output / "contrasts.csv"
if not scores.exists() or not contrasts.exists():
    raise FileNotFoundError("Expected scores.json and contrasts.csv were not produced")
print(f"Scores: {scores}")
print(f"Contrasts: {contrasts}")
display(pd.DataFrame(json.loads(scores.read_text())))
display(pd.read_csv(contrasts))


Scores: /kaggle/working/sia_pic_remaining_seeds/scores.json
Contrasts: /kaggle/working/sia_pic_remaining_seeds/contrasts.csv


,seed,kl,model,dataset,dimension,score
0,3,off,mrl,tweet,16,63.300034
1,3,off,mrl,tweet,32,65.392374
2,3,off,mrl,tweet,64,66.789620
3,3,off,mrl,tweet,128,68.265277
4,3,off,mrl,tweet,256,69.755919
...,...,...,...,...,...,...
331,5,on,mipic,stsb,64,67.176089
332,5,on,mipic,stsb,128,68.296512
333,5,on,mipic,stsb,256,69.639617
334,5,on,mipic,stsb,512,69.906893


,dataset,kl,dimension,quantity,seed_values,mean,ci_low,ci_high
0,banking77,off,16,interaction,"[3.6323392503730414, -5.43953945548594]",-0.903600,-58.538174,56.730974
1,banking77,off,16,sia_effect,"[2.371588248573847, -6.288457435411827]",-1.958435,-56.976591,53.059722
2,banking77,off,32,interaction,"[7.753956851008667, -5.82029620803662]",0.966830,-85.271789,87.205450
3,banking77,off,32,sia_effect,"[3.6392984301201423, -7.39129277395476]",-1.875997,-71.954472,68.202478
4,banking77,off,64,interaction,"[4.877649136475576, -6.415095849660389]",-0.768723,-72.512688,70.975242
...,...,...,...,...,...,...,...,...
85,tweet,on,512,interaction,"[1.1908614032461173, -0.9893160961239715]",0.100773,-13.750118,13.951663
86,tweet,on,512,sia_effect,"[-0.8823815017161962, -1.8657445315098045]",-1.374063,-7.621469,4.873343
87,tweet,on,768,full_width_gap,"[-3.4806430653717797, -3.529854447585194]",-3.505249,-3.817894,-3.192604
88,tweet,on,768,interaction,"[2.4526768401389205, -0.5488272365123521]",0.951925,-18.116938,20.020787
